# Train ModernBERT with an analytical-latency oracle

This is the recommended, end-to-end proof-of-concept notebook. It trains a rank-4 LoRA adapter on ModernBERT to predict the **fastest candidate that preserves the strongest model's quality**. Candidate LLMs are never loaded or timed here.

You will:

1. inspect a pre-collected LLMRouterBench release;
2. describe each candidate using model-card facts;
3. calculate latency from model size, architecture, precision, and prompt size;
4. build hindsight-oracle labels from pre-collected quality;
5. train ModernBERT with the oracle loss;
6. freeze the validation policy before opening the test split; and
7. export reports plus a reconstructable LoRA artifact.

> The result proves routing feasibility under stated analytical assumptions. It does not claim measured production latency.

## 1. Install the repository

Start Jupyter or Colab from the repository root. The base project dependencies include PyTorch, Transformers, PEFT, pandas, and scikit-learn. Restart the runtime if Jupyter asks after installation.

In [ ]:
%pip install -q -U -e ".[notebook]"

## 2. Imports, reproducibility, and paths

`DATA_ROOT` must point to an extracted LLMRouterBench checkout or release containing `results/bench/<dataset>/<split>/<model>/*.json`. Keep outputs outside the benchmark directory so the source evidence remains immutable.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display

from llm_router.modernbert_poc import (
    export_modernbert_oracle_poc,
    train_modernbert_oracle_poc,
)
from llm_router.oracle import oracle_choices
from llm_router.public_benchmark import (
    EconomicsScenario,
    ModelProfile,
    benchmark_inventory,
    export_public_benchmark,
    load_llmrouterbench,
    make_complete_panel,
    run_public_benchmark,
    simulate_economics,
    split_benchmark,
)
from llm_router.utils.training import seed_everything

SEED = 42
DATA_ROOT = Path("/content/LLMRouterBench")  # EDIT ME
OUTPUT_DIR = Path("reports_benchmark/modernbert_oracle_notebook")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

seed_everything(SEED)
assert DATA_ROOT.exists(), f"LLMRouterBench was not found at {DATA_ROOT}"
print({"device": DEVICE, "data_root": str(DATA_ROOT.resolve())})
if DEVICE == "cpu":
    print("Warning: CPU training works for a smoke test but will be slow. A GPU is recommended.")

## 3. Inspect available datasets and models

Do this before editing model profiles. The strings in `SELECTED_MODELS` must exactly match model directory names shown below. Choose at least two candidates and at least three datasets for a dataset-disjoint split.

In [ ]:
inventory = benchmark_inventory(DATA_ROOT)
assert not inventory.empty, "No LLMRouterBench result files were found."
inventory_summary = (
    inventory.groupby(["model", "dataset", "source_split"])
    .size()
    .rename("files")
    .reset_index()
)
display(inventory_summary)
print(f"Available models: {inventory.model.nunique()}")
print(f"Available datasets: {inventory.dataset.nunique()}")

## 4. Declare the candidate model profiles

Replace every placeholder. Parameter counts, active parameter counts, architecture, and precision should come from model cards or configuration files. For diffusion models, also record the inference-time denoising steps and generated block size.

Prices are set to zero because this notebook optimizes latency only. `effective_tflops`, memory bandwidth, fixed overhead, and router overhead are explicit POC assumptions—not measurements. Run sensitivity analyses with multiple reasonable scenarios before drawing a conclusion.

In [ ]:
MODEL_PROFILES = {
    "replace-with-small-ar-model-directory": {
        "parameters_billions": 1.5,
        "active_parameters_billions": 1.5,
        "architecture": "autoregressive",
        "weight_bits": 16,
    },
    "replace-with-diffusion-model-directory": {
        "parameters_billions": 1.5,
        "active_parameters_billions": 1.5,
        "architecture": "diffusion",
        "weight_bits": 16,
        "diffusion_steps": 8,
        "diffusion_block_size": 8,
    },
    "replace-with-strong-fallback-directory": {
        "parameters_billions": 7.0,
        "active_parameters_billions": 7.0,
        "architecture": "autoregressive",
        "weight_bits": 4,
    },
}

SELECTED_MODELS = tuple(MODEL_PROFILES)
assert len(SELECTED_MODELS) >= 2, "Routing requires at least two models."
assert not any(name.startswith("replace-with") for name in SELECTED_MODELS), (
    "Edit MODEL_PROFILES using exact directory names from the inventory above."
)
missing_models = sorted(set(SELECTED_MODELS) - set(inventory.model))
assert not missing_models, f"Models were not found in the benchmark: {missing_models}"
print(SELECTED_MODELS)

## 5. Load pre-collected quality and calculate analytical latency

The benchmark provides candidate answers and quality scores. `simulate_economics` calculates latency without loading those candidates. Expected output length is a bounded function of prompt length, so the policy cannot peek at a candidate's realized response length.

In [ ]:
profiles = tuple(
    ModelProfile(
        name=name,
        input_price_per_million=0.0,
        output_price_per_million=0.0,
        **settings,
    )
    for name, settings in MODEL_PROFILES.items()
)
scenario = EconomicsScenario(
    name="notebook-analytical-latency-poc",
    as_of=pd.Timestamp.utcnow().date().isoformat(),
    profiles=profiles,
    latency_method="analytical",
    effective_tflops=60.0,
    memory_bandwidth_gbps=900.0,
    fixed_model_overhead_s=0.015,
    router_overhead_s=0.004,
    output_base_tokens=24.0,
    output_tokens_per_prompt_token=0.20,
    output_min_tokens=16,
    output_max_tokens=256,
    notes="POC assumptions; not measured production latency.",
)

records = load_llmrouterbench(DATA_ROOT, models=SELECTED_MODELS)
simulated = simulate_economics(records, scenario)
panel = make_complete_panel(simulated, models=SELECTED_MODELS)
assert simulated.latency_source.eq("analytical").all()
print({"complete_prompts": len(panel.examples), "models": panel.models})
display(
    simulated.groupby("model")["simulated_latency_s"]
    .agg(["min", "median", "mean", "max"])
    .sort_values("mean")
)

### Leakage check

This deliberately changes every realized completion length. Analytical latency must remain identical because only prompt size and declared model/scenario properties are allowed to affect it.

In [ ]:
counterfactual_records = records.copy()
counterfactual_records["completion_tokens"] *= 100
counterfactual = simulate_economics(counterfactual_records, scenario)
assert np.allclose(
    simulated.simulated_latency_s,
    counterfactual.simulated_latency_s,
), "Analytical latency unexpectedly depends on realized completion length."
print("Passed: candidate completion tokens do not affect analytical latency.")

## 6. Freeze train, validation, and sealed-test splits

`dataset_ood` is the stronger POC: entire datasets are held out, so ModernBERT must generalize beyond training dataset identities. Use `random` only as an easier interpolation diagnostic. The fallback is selected from **training quality only**.

In [ ]:
split = split_benchmark(panel, mode="dataset_ood", seed=SEED)
split_summary = pd.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "prompts": [len(split.train), len(split.validation), len(split.test)],
        "datasets": [
            split.train_datasets,
            split.validation_datasets,
            split.test_datasets,
        ],
    }
)
display(split_summary)
assert set(split.train_datasets).isdisjoint(split.validation_datasets)
assert set(split.train_datasets).isdisjoint(split.test_datasets)
assert set(split.validation_datasets).isdisjoint(split.test_datasets)

## 7. Inspect the hindsight oracle before training

For each prompt, the oracle chooses the analytically fastest model whose observed score is no worse than the training-selected fallback. It is an unattainable upper bound because it uses answer outcomes. If it rarely leaves the fallback or saves little latency, no prompt-only router can prove much under this candidate panel and scenario.

In [ ]:
fallback_index = int(panel.score[split.train].mean(axis=0).argmax())
fallback_model = panel.models[fallback_index]
oracle_target = oracle_choices(
    panel.score, panel.latency, fallback_index=fallback_index
)
test_rows = np.arange(len(split.test))
test_oracle = oracle_target[split.test]
oracle_quality = panel.score[split.test][test_rows, test_oracle].mean()
fallback_quality = panel.score[split.test, fallback_index].mean()
oracle_latency = panel.latency[split.test][test_rows, test_oracle].mean()
fallback_latency = panel.latency[split.test, fallback_index].mean()

oracle_summary = pd.Series(
    {
        "fallback_model": fallback_model,
        "fallback_quality": fallback_quality,
        "oracle_quality": oracle_quality,
        "oracle_quality_retention": oracle_quality / max(fallback_quality, 1e-12),
        "fallback_latency_s": fallback_latency,
        "oracle_latency_s": oracle_latency,
        "oracle_latency_savings": 1 - oracle_latency / fallback_latency,
        "oracle_fallback_usage": np.mean(test_oracle == fallback_index),
    },
    name="sealed-test oracle upper bound",
)
display(oracle_summary.to_frame())
display(
    pd.Series(np.array(panel.models)[oracle_target[split.train]])
    .value_counts(normalize=True)
    .rename("train_oracle_rate")
)

## 8. Understand the training loss

ModernBERT emits one logit per candidate. Cross-entropy imitates the oracle and upweights prompts with larger safe speedups. Expected quality risk strongly penalizes probability assigned to models that score below the fallback. Expected latency regret penalizes probability assigned to models slower than the oracle.

$$
\mathcal L=(1+g_o)CE(z,o)+4\sum_m p_m d_m+\sum_m p_m r_m
$$

The loss is differentiable, but deployment remains conservative: a validation-selected confidence threshold, a 2% speed gate, and the aggregate quality-retention lower bound can all force fallback use.

## 9. Train ModernBERT

This downloads the pinned `nomic-ai/modernbert-embed-base`, freezes its base weights, inserts rank-4 LoRA adapters, and trains the LoRA parameters plus a small routing head. Five epochs are appropriate for the first POC; increase only after inspecting validation loss. Candidate LLMs are not loaded.

In [ ]:
training = train_modernbert_oracle_poc(
    panel,
    split,
    epochs=5,
    batch_size=8,
    learning_rate=1e-4,
    quality_epsilon=0.0,
    device=DEVICE,
)
display(training.history)
print({"training_seconds": round(training.training_seconds, 1)})
assert training.probabilities.shape == panel.score.shape
assert np.allclose(training.probabilities.sum(axis=1), 1.0, atol=1e-4)

## 10. Select the validation policy, then open the sealed test

The selector searches confidence thresholds using validation only. A proposed alternative must be at least 2% faster analytically. The router activates only when validation has positive net savings after assumed router overhead and its one-sided 95% quality-retention lower bound is at least 98%. Test quality is consumed only after that policy is frozen.

In [ ]:
result = run_public_benchmark(
    panel,
    split,
    objective="latency",
    minimum_quality_retention=0.98,
    confidence=0.95,
    minimum_predicted_savings=0.02,
    router_overhead_s=scenario.router_overhead_s,
    seed=SEED,
    routing_probabilities=training.probabilities,
    probability_kind="oracle",
    router_name="modernbert_oracle_router",
)
assert result.fallback_model == fallback_model
display(result.threshold_search)
display(result.summary)
print(
    {
        "router_active": result.router_active,
        "selected_threshold": result.selected_threshold,
        "fallback_model": result.fallback_model,
    }
)

### Interpret the result

A convincing POC has oracle headroom, non-trivial ModernBERT alternative usage, at least 98% quality retention at the one-sided 95% lower bound, positive analytical savings after router overhead, and similar conclusions under reasonable scenario changes.

If `router_active` is false, that is a valid result. Compare the `outcome_oracle` and `modernbert_oracle_router` rows:

- weak oracle savings means the candidate panel or assumptions offer little headroom;
- strong oracle savings but fallback-only routing points to prediction/generalization difficulty;
- good random-split results but weak dataset-OOD results indicate dataset-specific overfitting;
- good quality but weak savings suggests the confidence threshold or router-overhead assumption is too costly.

## 11. Export reproducible reports and the router artifact

The report manifest records every analytical assumption. The router artifact contains the LoRA adapter, routing head, tokenizer, base encoder revision, selected confidence threshold, fallback, and deployment guard state.

In [ ]:
report_dir = export_public_benchmark(result, scenario, OUTPUT_DIR)
artifact_dir = export_modernbert_oracle_poc(
    training,
    panel.models,
    report_dir / "modernbert_router",
    selected_threshold=result.selected_threshold,
    router_active=result.router_active,
    minimum_predicted_savings=0.02,
)
print({"reports": str(report_dir.resolve()), "artifact": str(artifact_dir.resolve())})

## 12. Final POC checklist

Before presenting the result, confirm:

- all model profiles are sourced and not placeholders;
- no candidate inference was run to construct latency;
- the completion-length leakage assertion passed;
- fallback selection used training quality only;
- confidence and activation were selected on validation only;
- the test split was opened once after policy freeze;
- results are described as analytical or simulated latency, never measured latency; and
- the conclusion survives multiple reasonable hardware and diffusion-step scenarios.